In [14]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)

# 90 day date range
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 3, 31)
dates = [start_date + timedelta(days=i) for i in range(90)]

print("Date range:", dates[0].date(), "to", dates[-1].date())
print("Total days:", len(dates))
print("Setup complete ✓")

Date range: 2024-01-01 to 2024-03-30
Total days: 90
Setup complete ✓


In [15]:
# Policy categories and their base daily flag rates
CATEGORIES = {
    "harassment":     {"base_rate": 120, "trend": +0.8},  # trending up
    "hate_speech":    {"base_rate": 45,  "trend": +0.3},
    "spam":           {"base_rate": 90,  "trend": -0.5},  # trending down
    "sexual_content": {"base_rate": 30,  "trend": +0.1},
    "self_harm":      {"base_rate": 20,  "trend": +0.2},
    "fraud":          {"base_rate": 55,  "trend": -0.2},
}

SEVERITIES = ["low", "medium", "high", "critical"]
SEVERITY_WEIGHTS = [0.45, 0.35, 0.15, 0.05]

ACTIONS = ["allow", "downrank", "human_review", "auto_block"]

print("Categories:", list(CATEGORIES.keys()))
print("Total base flags per day:", sum(c["base_rate"] for c in CATEGORIES.values()))
print("Taxonomy defined ✓")

Categories: ['harassment', 'hate_speech', 'spam', 'sexual_content', 'self_harm', 'fraud']
Total base flags per day: 360
Taxonomy defined ✓


In [16]:
# Define 5 coordinated attack events (ground truth)
attack_events = [
    {
        "attack_id": "ATK001",
        "start_date": datetime(2024, 1, 15),
        "duration_days": 2,
        "category": "harassment",
        "num_accounts": 45,
        "pattern": "Coordinated harassment targeting female journalists",
        "severity": "high"
    },
    {
        "attack_id": "ATK002", 
        "start_date": datetime(2024, 1, 28),
        "duration_days": 1,
        "category": "hate_speech",
        "num_accounts": 30,
        "pattern": "Hate speech spike around political event",
        "severity": "critical"
    },
    {
        "attack_id": "ATK003",
        "start_date": datetime(2024, 2, 10),
        "duration_days": 3,
        "category": "spam",
        "num_accounts": 120,
        "pattern": "Spam campaign promoting crypto scam links",
        "severity": "medium"
    },
    {
        "attack_id": "ATK004",
        "start_date": datetime(2024, 2, 25),
        "duration_days": 2,
        "category": "harassment",
        "num_accounts": 60,
        "pattern": "Targeted harassment campaign against activist group",
        "severity": "critical"
    },
    {
        "attack_id": "ATK005",
        "start_date": datetime(2024, 3, 18),
        "duration_days": 1,
        "category": "fraud",
        "num_accounts": 35,
        "pattern": "Fraudulent account network promoting fake giveaways",
        "severity": "high"
    },
]

# Build lookup: date + category -> attack_id
attack_lookup = {}
for atk in attack_events:
    for d in range(atk["duration_days"]):
        day = atk["start_date"] + timedelta(days=d)
        attack_lookup[(day.date(), atk["category"])] = atk["attack_id"]

print(f"Attack events defined: {len(attack_events)}")
for atk in attack_events:
    print(f"  {atk['attack_id']}: {atk['category']} | {atk['start_date'].date()} | {atk['num_accounts']} accounts")

Attack events defined: 5
  ATK001: harassment | 2024-01-15 | 45 accounts
  ATK002: hate_speech | 2024-01-28 | 30 accounts
  ATK003: spam | 2024-02-10 | 120 accounts
  ATK004: harassment | 2024-02-25 | 60 accounts
  ATK005: fraud | 2024-03-18 | 35 accounts


In [17]:
# Generate flags row by row
flags = []
flag_id = 1

for date in dates:
    day_of_week = date.weekday()  # 0=Monday, 6=Sunday
    is_weekend = day_of_week >= 4  # Fri, Sat, Sun
    day_num = (date - start_date).days  # 0 to 89

    for category, config in CATEGORIES.items():
        # Base rate + trend over time
        base = config["base_rate"] + (config["trend"] * day_num)

        # Weekend spike — 25% more flags
        if is_weekend:
            base *= 1.25

        # Attack event spike — 3x more flags on attack days
        is_attack = (date.date(), category) in attack_lookup
        if is_attack:
            base *= 5.0

        # Add daily noise
        if is_attack:
            daily_count = max(5, int(np.random.normal(base, base * 0.05)))
        else:
            daily_count = max(5, int(np.random.normal(base, base * 0.15)))

        for _ in range(daily_count):
            severity = np.random.choice(SEVERITIES, p=SEVERITY_WEIGHTS)

            # Severity skews higher during attacks
            if is_attack:
                severity = np.random.choice(SEVERITIES, p=[0.10, 0.30, 0.40, 0.20])

            # Action based on severity
            if severity == "critical":
                action = np.random.choice(["auto_block", "human_review"], p=[0.85, 0.15])
            elif severity == "high":
                action = np.random.choice(["auto_block", "human_review"], p=[0.50, 0.50])
            elif severity == "medium":
                action = np.random.choice(["human_review", "downrank"], p=[0.60, 0.40])
            else:
                action = np.random.choice(["downrank", "allow"], p=[0.40, 0.60])

            # ML score — gradual drift from 0.86 to 0.78 over 90 days
            base_accuracy = 0.86 - (day_num / 90) * 0.08
            ml_score = np.clip(np.random.normal(base_accuracy, 0.05), 0.5, 1.0)

            # Resolution time in hours
            if severity == "critical":
                resolution_hrs = round(np.random.exponential(8), 1)
            elif severity == "high":
                resolution_hrs = round(np.random.exponential(16), 1)
            elif severity == "medium":
                resolution_hrs = round(np.random.exponential(24), 1)
            else:
                resolution_hrs = round(np.random.exponential(48), 1)

            # Reviewer assignment (20 reviewers)
            reviewer_id = f"REV{np.random.randint(1, 21):03d}"

            flags.append({
                "flag_id": f"F{flag_id:06d}",
                "date": date.date(),
                "category": category,
                "severity": severity,
                "action": action,
                "reviewer_id": reviewer_id,
                "resolution_hrs": resolution_hrs,
                "ml_score": round(ml_score, 4),
                "coordinated_flag": is_attack,
                "attack_id": attack_lookup.get((date.date(), category), None)
            })
            flag_id += 1

df_flags = pd.DataFrame(flags)

print("Flags generated:", len(df_flags))
print("\nCategory distribution:")
print(df_flags["category"].value_counts())
print("\nSeverity distribution:")
print(df_flags["severity"].value_counts())
print("\nCoordinated flags:", df_flags["coordinated_flag"].sum())

Flags generated: 41935

Category distribution:
category
harassment        17505
spam               7527
hate_speech        5881
fraud              4705
sexual_content     3387
self_harm          2930
Name: count, dtype: int64

Severity distribution:
severity
low         17176
medium      14497
high         7411
critical     2851
Name: count, dtype: int64

Coordinated flags: 4714


In [18]:
import matplotlib.pyplot as plt

# Check attack days have higher flag counts
daily_total = df_flags.groupby("date").size().reset_index(name="total_flags")
daily_total["date"] = pd.to_datetime(daily_total["date"])

# Mark attack dates
attack_dates = set()
for atk in attack_events:
    for d in range(atk["duration_days"]):
        attack_dates.add((atk["start_date"] + timedelta(days=d)).date())

daily_total["is_attack"] = daily_total["date"].dt.date.isin(attack_dates)

plt.figure(figsize=(14, 4))
plt.plot(daily_total["date"], daily_total["total_flags"], color="steelblue", linewidth=1.5, label="Daily flags")
attack_days = daily_total[daily_total["is_attack"]]
plt.scatter(attack_days["date"], attack_days["total_flags"], color="red", zorder=5, label="Attack days", s=60)
plt.title("Daily Flag Volume — Attack Days Highlighted")
plt.xlabel("Date")
plt.ylabel("Total Flags")
plt.legend()
plt.tight_layout()
plt.show()

print("Average flags on normal days:", round(daily_total[~daily_total["is_attack"]]["total_flags"].mean(), 1))
print("Average flags on attack days:", round(daily_total[daily_total["is_attack"]]["total_flags"].mean(), 1))

Average flags on normal days: 424.6
Average flags on attack days: 838.1


C:\Users\91986\AppData\Local\Temp\ipykernel_18708\4031733982.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# Check drift is gradual and realistic
daily_ml = df_flags.groupby("date")["ml_score"].mean().reset_index()
daily_ml["date"] = pd.to_datetime(daily_ml["date"])
daily_ml["day_num"] = range(len(daily_ml))

# Rolling 7-day average
daily_ml["rolling_avg"] = daily_ml["ml_score"].rolling(7, min_periods=3).mean()

plt.figure(figsize=(14, 4))
plt.plot(daily_ml["date"], daily_ml["ml_score"], color="lightblue", linewidth=1, alpha=0.7, label="Daily ML score")
plt.plot(daily_ml["date"], daily_ml["rolling_avg"], color="steelblue", linewidth=2.5, label="7-day rolling avg")
plt.axhline(y=0.86, color="green", linestyle="--", alpha=0.7, label="Baseline (0.86)")
plt.axhline(y=0.78, color="red", linestyle="--", alpha=0.7, label="Target drift (0.78)")
plt.title("Classifier Drift — ML Score Over Time")
plt.xlabel("Date")
plt.ylabel("Average ML Score")
plt.legend()
plt.ylim(0.70, 0.92)
plt.tight_layout()
plt.show()

print("Day 1 avg ML score:", round(daily_ml["ml_score"].iloc[:7].mean(), 4))
print("Day 90 avg ML score:", round(daily_ml["ml_score"].iloc[-7:].mean(), 4))
print("Total drift:", round(daily_ml["ml_score"].iloc[:7].mean() - daily_ml["ml_score"].iloc[-7:].mean(), 4))

Day 1 avg ML score: 0.8567
Day 90 avg ML score: 0.7845
Total drift: 0.0723


C:\Users\91986\AppData\Local\Temp\ipykernel_18708\1026227990.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Generate reviewers table
reviewers = []
for i in range(1, 21):
    reviewers.append({
        "reviewer_id": f"REV{i:03d}",
        "name": f"Reviewer_{i:02d}",
        "team": np.random.choice(["APAC", "EMEA", "AMER"], p=[0.3, 0.3, 0.4]),
        "capacity_per_day": np.random.choice([40, 50, 60, 70], p=[0.2, 0.4, 0.3, 0.1]),
        "seniority": np.random.choice(["junior", "mid", "senior"], p=[0.4, 0.4, 0.2])
    })

df_reviewers = pd.DataFrame(reviewers)

# Generate attacks summary table
df_attacks = pd.DataFrame(attack_events)
df_attacks["start_date"] = df_attacks["start_date"].dt.date
df_attacks["end_date"] = df_attacks.apply(
    lambda r: (datetime.combine(r["start_date"], datetime.min.time()) + 
               timedelta(days=r["duration_days"]-1)).date(), axis=1
)

# Save all tables
os.makedirs("../data", exist_ok=True)
df_flags.to_csv("../data/flags.csv", index=False)
df_reviewers.to_csv("../data/reviewers.csv", index=False)
df_attacks.to_csv("../data/attacks.csv", index=False)

print("Saved:")
print(f"  flags.csv      — {len(df_flags):,} rows")
print(f"  reviewers.csv  — {len(df_reviewers)} rows")
print(f"  attacks.csv    — {len(df_attacks)} rows")
print()
print("Reviewer teams:")
print(df_reviewers["team"].value_counts())
print()
print("Attacks table:")
print(df_attacks[["attack_id", "category", "start_date", "duration_days", "num_accounts", "severity"]])

Saved:
  flags.csv      — 41,935 rows
  reviewers.csv  — 20 rows
  attacks.csv    — 5 rows

Reviewer teams:
team
EMEA    7
APAC    7
AMER    6
Name: count, dtype: int64

Attacks table:
  attack_id     category  start_date  duration_days  num_accounts  severity
0    ATK001   harassment  2024-01-15              2            45      high
1    ATK002  hate_speech  2024-01-28              1            30  critical
2    ATK003         spam  2024-02-10              3           120    medium
3    ATK004   harassment  2024-02-25              2            60  critical
4    ATK005        fraud  2024-03-18              1            35      high


In [21]:
# Final data health check
print("=== DATA HEALTH CHECK ===\n")

print("flags.csv")
print(f"  Rows: {len(df_flags):,}")
print(f"  Date range: {df_flags['date'].min()} to {df_flags['date'].max()}")
print(f"  Nulls: {df_flags.isnull().sum().sum()}")
print(f"  Coordinated rows: {df_flags['coordinated_flag'].sum():,} ({df_flags['coordinated_flag'].mean()*100:.1f}%)")
print()

print("Category trends (first 30 days vs last 30 days):")
early = df_flags[df_flags["date"] <= pd.to_datetime("2024-01-31").date()]
late = df_flags[df_flags["date"] >= pd.to_datetime("2024-03-01").date()]
for cat in CATEGORIES.keys():
    early_count = len(early[early["category"] == cat])
    late_count = len(late[late["category"] == cat])
    direction = "↑" if late_count > early_count else "↓"
    print(f"  {cat:20s}: {early_count:4d} → {late_count:4d} {direction}")

print()
print("SLA risk (HIGH/CRITICAL cases > 24hrs):")
high_crit = df_flags[df_flags["severity"].isin(["high", "critical"])]
sla_breach = high_crit[high_crit["resolution_hrs"] > 24]
print(f"  High/Critical cases: {len(high_crit):,}")
print(f"  SLA breaches (>24hr): {len(sla_breach):,} ({len(sla_breach)/len(high_crit)*100:.1f}%)")

print()
print("=== DATA READY ✓ ===")

=== DATA HEALTH CHECK ===

flags.csv
  Rows: 41,935
  Date range: 2024-01-01 to 2024-03-30
  Nulls: 37221
  Coordinated rows: 4,714 (11.2%)

Category trends (first 30 days vs last 30 days):
  harassment          : 5366 → 5969 ↑
  hate_speech         : 1836 → 2208 ↑
  spam                : 2700 → 1718 ↓
  sexual_content      : 1080 → 1234 ↑
  self_harm           :  755 → 1200 ↑
  fraud               : 1704 → 1471 ↓

SLA risk (HIGH/CRITICAL cases > 24hrs):
  High/Critical cases: 10,262
  SLA breaches (>24hr): 1,850 (18.0%)

=== DATA READY ✓ ===


In [22]:
print("Null counts by column:")
print(df_flags.isnull().sum())

Null counts by column:
flag_id                 0
date                    0
category                0
severity                0
action                  0
reviewer_id             0
resolution_hrs          0
ml_score                0
coordinated_flag        0
attack_id           37221
dtype: int64
